<a href="https://colab.research.google.com/github/Gilnei732/.github-workflows-atualizar-acervo.yml/blob/main/Transcricao_Video_Voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
pip install -q yt-dlp python-docx google-genai

In [2]:
%%bash
pip install -q openai-whisper

### Passo 2: Baixar o áudio do vídeo do YouTube
Vamos baixar apenas o áudio do vídeo utilizando o `yt-dlp` no formato MP3.

In [3]:
import yt_dlp

video_url = 'https://www.youtube.com/watch?v=SVg2TmT8Q5s'
output_audio = 'audio_input'

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': output_audio,
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])

import os
audio_file = f"{output_audio}.mp3"
print(f"Áudio baixado com sucesso: {audio_file}" if os.path.exists(audio_file) else "Erro no download.")

[youtube] Extracting URL: https://www.youtube.com/watch?v=SVg2TmT8Q5s
[youtube] SVg2TmT8Q5s: Downloading webpage


[youtube] SVg2TmT8Q5s: Downloading visionos player API JSON
[youtube] SVg2TmT8Q5s: Downloading m3u8 information
[info] SVg2TmT8Q5s: Downloading 1 format(s): 251-2
[download] Destination: audio_input
[download] 100% of   12.81MiB in 00:00:01 at 12.43MiB/s  
[ExtractAudio] Destination: audio_input.mp3
Deleting original file audio_input (pass -k to keep)
Áudio baixado com sucesso: audio_input.mp3


### Passo 3: Transcrever o áudio com o Whisper

Para fazer a transcrição, usaremos o modelo Whisper que possui excelente suporte para áudio e gera transcrições precisas em português.

Pelo Colab e Gemini é Preciso Inserir uma chave - Deve ser copiada no ìcone Lateral de Chave (aqui ao lado)

*Nota: Certifique-se de configurar a sua chave da API do Gemini (`GOOGLE_API_KEY`) no menu de segredos (ícone de chave 🔑 no menu lateral do Colab).*

In [ ]:
import whisper

print("Carregando o modelo Whisper (isso pode levar alguns instantes)...")
# Usando o modelo 'small' para um bom equilíbrio entre velocidade e precisão em português
model = whisper.load_model("small")

print("Iniciando a transcrição local do áudio...")
result = model.transcribe(audio_file, verbose=True, language="pt")

transcricao = result["text"]
print("\n--- Transcrição Concluída com Sucesso! ---")

Carregando o modelo Whisper (isso pode levar alguns instantes)...


100%|████████████████████████████████████████| 461M/461M [00:04<00:00, 114MiB/s]


Iniciando a transcrição local do áudio...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[00:00.000 --> 00:02.000]  Isso é importante chegando agora pessoal.
[00:02.000 --> 00:05.120]  Agora, cinco horas em ponto, Flávio Bolsonaro,
[00:05.120 --> 00:08.360]  o senador disse que foi intimado pela Polícia Federal
[00:08.360 --> 00:10.800]  a entregar o passaporte dele.
[00:10.800 --> 00:13.120]  O senador disse que foi intimado
[00:13.120 --> 00:15.200]  em um vídeo publicado nas redes sociais.
[00:15.200 --> 00:17.240]  Ele afirmou que a intimação aconteceu.
[00:17.240 --> 00:19.440]  Depois da visita ao pai dele,
[00:19.440 --> 00:22.800]  Jair Bolsonaro, que a gente falava aqui mais cedo às 11 da manhã,
[00:22.800 --> 00:25.480]  foi visitar o pai em prisão domiciliar.
[00:25.480 --> 00:26.960]  A gente está separando esse trecho,
[00:26.960 --> 00:28.080]  mas ele diz abrir aspas.
[00:28.080 --> 00:31.840]  A Polícia Federal do Lula estava na porta da casa dele.
[00:31.840 --> 00:35.880]  E no meio da rua, eu fui intimado a entregar os meus passaportes.
[00:35.880 --> 00

Fazendo o Download do arquivo Docx Completo

# Agora fazendo pelo Colab/Gemini ... Pra isso precisa da chave API - que deve ser copiada na chave ao lado esquerdo.

In [ ]:
from google import genai
from google.colab import userdata
import time
import getpass

# Tenta obter a chave dos Segredos do Colab, se não existir, solicita ao usuário
try:
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    print("Chave 'GOOGLE_API_KEY' não encontrada nos Segredos do Colab.")
    api_key = getpass.getpass("Por favor, insira sua Gemini API Key: ")

client = genai.Client(api_key=api_key)

print("Fazendo upload do arquivo de áudio para a API do Gemini...")
audio_upload = client.files.upload(file=audio_file)

# Aguarda o processamento do arquivo de áudio na nuvem se necessário
while audio_upload.state.name == "PROCESSING":
    print("Processando arquivo de áudio...")
    time.sleep(5)
    audio_upload = client.files.get(name=audio_upload.name)

print("Iniciando a transcrição completa do áudio...")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[
        audio_upload,
        "Transcreva todo o áudio do arquivo fornecido de forma literal, completa e em português, mantendo a fidelidade das falas."
    ]
)

transcricao = response.text
print("\n--- Transcrição Concluída! Primeiros 500 caracteres: ---")
print(transcricao[:500] + "...")

In [ ]:
import sys
# Garante a atualização dos pacotes no ambiente atual do Python
!{sys.executable} -m pip install -q python-docx

from docx import Document
from google.colab import files

# Criando um novo documento Word
doc = Document()
doc.add_heading('Histórico do Notebook: Transcrição de Vídeo', level=1)

# Adicionando a primeira seção de Instalações
doc.add_heading('1. Instalação de Dependências', level=2)
doc.add_paragraph('Instalação do yt-dlp, python-docx, google-genai e openai-whisper:')
code1 = doc.add_paragraph()
run1 = code1.add_run('pip install -q yt-dlp python-docx google-genai openai-whisper')
run1.font.name = 'Courier New'

# Passo 2: Baixar o áudio
doc.add_heading('2. Download do Áudio do YouTube', level=2)
doc.add_paragraph('Código utilizado para baixar o áudio do link do YouTube utilizando yt-dlp:')
code2 = doc.add_paragraph()
code2_text = """import yt_dlp

video_url = 'https://www.youtube.com/watch?v=SVg2TmT8Q5s'
output_audio = 'audio_input'

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': output_audio,
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])
"""
run2 = code2.add_run(code2_text)
run2.font.name = 'Courier New'

# Saída do download
doc.add_heading('Saída da Execução (Download do Áudio):', level=3)
out2_p = doc.add_paragraph()
out2_run = out2_p.add_run('[youtube] Extracting URL: https://www.youtube.com/watch?v=SVg2TmT8Q5s\n[youtube] Downloading webpage\n[download] Destination: audio_input\n[ExtractAudio] Destination: audio_input.mp3\nÁudio baixado com sucesso: audio_input.mp3')
out2_run.italic = True

# Passo 3: Transcrição Local via Whisper
doc.add_heading('3. Transcrição Sem Chave de API (Whisper Local)', level=2)
doc.add_paragraph('Carregando o modelo Whisper e executando a transcrição do áudio localmente:')
code3 = doc.add_paragraph()
code3_text = """import whisper

print("Carregando o modelo Whisper...")
model = whisper.load_model("small")

print("Iniciando a transcrição local do áudio...")
result = model.transcribe(audio_file, verbose=True, language="pt")
transcricao = result["text"]
"""
run3 = code3.add_run(code3_text)
run3.font.name = 'Courier New'

# Saída da Transcrição
doc.add_heading('Saída da Execução (Transcrição Obtida):', level=3)
out3_p = doc.add_paragraph()
# Caso a variável local já exista no ambiente, usamos o valor dela, senão uma prévia de contexto
try:
    texto_transcrito = transcricao
except NameError:
    texto_transcrito = "Transcrição não localizada na memória. Rode a célula de transcrição local antes de gerar este arquivo."
out3_run = out3_p.add_run(texto_transcrito)

# Salvando e baixando o arquivo docx
filename = 'historico_e_transcricao_notebook.docx'
doc.save(filename)
print(f"Documento {filename} gerado com sucesso!")
files.download(filename)

### Passo 4: Criar o arquivo DOCX e fazer o download
Agora, criaremos um documento do Word contendo o texto transcrito e ofereceremos o download automático dele.